In [66]:
import torch

In [67]:
def conv2d(x, weight, bias):
    # x: (C, H, W)
    # weight: (F, C, kH, kW)
    
    C, H, W = x.shape
    F, _, kH, kW = weight.shape
    
    out_H = H - kH + 1
    out_W = W - kW + 1
    
    out = torch.zeros((F, out_H, out_W))
    
    for f in range(F):
        for i in range(out_H):
            for j in range(out_W):
                region = x[:, i:i+kH, j:j+kW]
                out[f, i, j] = torch.sum(region * weight[f]) + bias[f]
    
    return out

In [68]:
def relu(x):
    return torch.maximum(x, torch.tensor(0.0))

In [69]:
def max_pool2d(x, size=2):
    # x: (C, H, W)
    C, H, W = x.shape
    
    out_H = H // size
    out_W = W // size
    
    out = torch.zeros((C, out_H, out_W))
    
    for c in range(C):
        for i in range(out_H):
            for j in range(out_W):
                region = x[c,
                           i*size:(i+1)*size,
                           j*size:(j+1)*size]
                out[c, i, j] = torch.max(region)
    
    return out

In [70]:
def flatten(x):
    return x.view(-1)

In [71]:
def linear(x, weight, bias):
    return x @ weight.T + bias

In [72]:
def softmax(x):
    exp = torch.exp(x)
    return exp / torch.sum(exp)

In [73]:
def cross_entropy(probs, label):
    return -torch.log(probs[label])

In [74]:
# Conv layer
conv_weight = torch.randn(8, 1, 3, 3, requires_grad=True)  # 8 filters
conv_bias = torch.randn(8, requires_grad=True)

# After conv (28 → 26), pooling (26 → 13)
fc_input_size = 8 * 13 * 13

# FC layer
fc_weight = torch.randn(10, fc_input_size, requires_grad=True)
fc_bias = torch.randn(10, requires_grad=True)

In [76]:
def forward(x):
    z1 = conv2d(x, conv_W, conv_b)
    a1 = relu(z1)
    
    f = flatten(a1)
    
    z2 = linear(f, fc_W, fc_b)
    probs = softmax(z2)
    
    cache = (x, z1, a1, f, z2, probs)
    return probs, cache

In [77]:
def backward(cache, label):
    global conv_W, conv_b, fc_W, fc_b
    
    x, z1, a1, f, z2, probs = cache
    
    # ---- FC Layer ----
    
    dz2 = probs.clone()
    dz2[label] -= 1   # dL/dz2
    
    dW_fc = dz2.unsqueeze(1) @ f.unsqueeze(0)  # outer product
    db_fc = dz2
    
    df = dz2 @ fc_W  # backprop to flatten
    
    # ---- Flatten ----
    
    da1 = df.view(a1.shape)
    
    # ---- ReLU ----
    
    dz1 = da1.clone()
    dz1[z1 <= 0] = 0
    
    # ---- Conv Layer ----
    
    F, H, W = dz1.shape
    _, C, kH, kW = conv_W.shape
    
    dW_conv = torch.zeros_like(conv_W)
    db_conv = torch.zeros_like(conv_b)
    
    for f_idx in range(F):
        for i in range(H):
            for j in range(W):
                region = x[:, i:i+kH, j:j+kW]
                dW_conv[f_idx] += dz1[f_idx, i, j] * region
                db_conv[f_idx] += dz1[f_idx, i, j]
    
    return dW_fc, db_fc, dW_conv, db_conv

In [78]:
def update(dW_fc, db_fc, dW_conv, db_conv, lr=0.001):
    global conv_W, conv_b, fc_W, fc_b
    
    fc_W -= lr * dW_fc
    fc_b -= lr * db_fc
    
    conv_W -= lr * dW_conv
    conv_b -= lr * db_conv

In [79]:
conv_W = torch.randn(4, 1, 3, 3) * 0.1
conv_b = torch.zeros(4)

fc_W = torch.randn(10, 4*26*26) * 0.1
fc_b = torch.zeros(10)

In [80]:
def train_step(x, label):
    probs, cache = forward(x)
    
    loss = -torch.log(probs[label])
    
    grads = backward(cache, label)
    update(*grads)
    
    return loss.item()

In [ ]:
def cross_entropy_loss(logits, label):
    exp = torch.exp(logits)
    probs = exp / torch.sum(exp)
    return -torch.log(probs[label])

In [82]:
x = torch.randn(1, 28, 28)
label = 2

loss = train_step(x, label)
print("Loss:", loss)

Loss: 0.6290484666824341


In [88]:
import torch

# ==============================
# Utility: im2col (vectorized conv)
# ==============================
def im2col(x, kH, kW):
    C, H, W = x.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    
    cols = []
    for i in range(out_H):
        for j in range(out_W):
            patch = x[:, i:i+kH, j:j+kW].reshape(-1)
            cols.append(patch)
    
    return torch.stack(cols).T  # (C*kH*kW, out_H*out_W)

# ==============================
# Conv Forward (vectorized)
# ==============================
def conv_forward(x, W, b):
    F, C, kH, kW = W.shape
    
    x_col = im2col(x, kH, kW)  # (C*kH*kW, N)
    W_col = W.view(F, -1)      # (F, C*kH*kW)
    
    out = W_col @ x_col + b.unsqueeze(1)
    
    out_H = x.shape[1] - kH + 1
    out_W = x.shape[2] - kW + 1
    
    return out.view(F, out_H, out_W), x_col

# ==============================
# Conv Backward (vectorized)
# ==============================
def conv_backward(dout, x, x_col, W):
    F, C, kH, kW = W.shape
    
    dout_flat = dout.view(F, -1)
    
    dW = dout_flat @ x_col.T
    dW = dW.view(W.shape)
    
    db = torch.sum(dout_flat, dim=1)
    
    W_col = W.view(F, -1)
    dx_col = W_col.T @ dout_flat
    
    dx = torch.zeros_like(x)
    
    out_H = x.shape[1] - kH + 1
    out_W = x.shape[2] - kW + 1
    
    idx = 0
    for i in range(out_H):
        for j in range(out_W):
            patch = dx_col[:, idx].view(C, kH, kW)
            dx[:, i:i+kH, j:j+kW] += patch
            idx += 1
    
    return dx, dW, db

# ==============================
# ReLU
# ==============================
def relu_forward(x):
    return torch.maximum(x, torch.tensor(0.0))

def relu_backward(dout, x):
    dx = dout.clone()
    dx[x <= 0] = 0
    return dx

# ==============================
# MaxPool Forward
# ==============================
def maxpool_forward(x, size=2):
    C, H, W = x.shape
    out_H = H // size
    out_W = W // size
    
    out = torch.zeros((C, out_H, out_W))
    mask = torch.zeros_like(x)
    
    for c in range(C):
        for i in range(out_H):
            for j in range(out_W):
                region = x[c, i*size:(i+1)*size, j*size:(j+1)*size]
                max_val = torch.max(region)
                out[c, i, j] = max_val
                
                mask_region = (region == max_val)
                mask[c, i*size:(i+1)*size, j*size:(j+1)*size] = mask_region
    
    return out, mask

# ==============================
# MaxPool Backward
# ==============================
def maxpool_backward(dout, mask, size=2):
    C, H, W = mask.shape
    dx = torch.zeros_like(mask)
    
    out_H = dout.shape[1]
    out_W = dout.shape[2]
    
    for c in range(C):
        for i in range(out_H):
            for j in range(out_W):
                dx[c, i*size:(i+1)*size, j*size:(j+1)*size] += \
                    dout[c, i, j] * mask[c, i*size:(i+1)*size, j*size:(j+1)*size]
    
    return dx

# ==============================
# Flatten
# ==============================
def flatten(x):
    return x.view(-1)

def flatten_backward(dout, shape):
    return dout.view(shape)

# ==============================
# Linear
# ==============================
def linear_forward(x, W, b):
    return x @ W.T + b

def linear_backward(dout, x, W):
    dW = dout.unsqueeze(1) @ x.unsqueeze(0)
    db = dout
    dx = dout @ W
    return dx, dW, db

# ==============================
# Softmax + Loss
# ==============================
def softmax(x):
    exp = torch.exp(x - torch.max(x))
    return exp / torch.sum(exp)

def cross_entropy(probs, label):
    return -torch.log(probs[label])

# ==============================
# MODEL PARAMETERS
# ==============================
conv_W = torch.randn(8, 1, 3, 3) * 0.1
conv_b = torch.zeros(8)

fc_W = torch.randn(10, 8*13*13) * 0.1
fc_b = torch.zeros(10)

# ==============================
# TRAIN STEP
# ==============================
def train_step_debug(x, label, lr=0.001):
    global conv_W, conv_b, fc_W, fc_b

    print("\n===== INPUT =====")
    print("x shape:", x.shape)

    # ===== FORWARD =====
    conv_out, x_col = conv_forward(x, conv_W, conv_b)
    print("\n[Conv]")
    print("conv_out shape:", conv_out.shape)

    relu_out = relu_forward(conv_out)
    print("\n[ReLU]")
    print("relu_out min/max:", relu_out.min().item(), relu_out.max().item())

    pool_out, mask = maxpool_forward(relu_out)
    print("\n[MaxPool]")
    print("pool_out shape:", pool_out.shape)

    flat = flatten(pool_out)
    print("\n[Flatten]")
    print("flat shape:", flat.shape)

    logits = linear_forward(flat, fc_W, fc_b)
    print("\n[Linear]")
    print("logits shape:", logits.shape)
    print("logits:", logits)

    probs = softmax(logits)
    print("\n[Softmax]")
    print("probs:", probs)
    print("sum of probs (should be 1):", probs.sum().item())

    loss = cross_entropy(probs, label)
    print("\n[Loss]")
    print("loss:", loss.item())

    # ===== BACKWARD =====
    print("\n===== BACKWARD =====")

    dlogits = probs.clone()
    dlogits[label] -= 1
    print("dlogits:", dlogits)

    dflat, dW_fc, db_fc = linear_backward(dlogits, flat, fc_W)
    print("\n[Linear Backward]")
    print("dflat shape:", dflat.shape)

    dpool = flatten_backward(dflat, pool_out.shape)
    print("\n[Flatten Backward]")
    print("dpool shape:", dpool.shape)

    drelu = maxpool_backward(dpool, mask)
    print("\n[MaxPool Backward]")
    print("drelu shape:", drelu.shape)

    dconv = relu_backward(drelu, conv_out)
    print("\n[ReLU Backward]")
    print("dconv shape:", dconv.shape)

    dx, dW_conv, db_conv = conv_backward(dconv, x, x_col, conv_W)
    print("\n[Conv Backward]")
    print("dx shape:", dx.shape)

    # ===== UPDATE =====
    conv_W -= lr * dW_conv
    conv_b -= lr * db_conv

    fc_W -= lr * dW_fc
    fc_b -= lr * db_fc

    return loss.item()

# ==============================
# TEST RUN
# ==============================
x = torch.randn(1, 28, 28)
label = 3

loss = train_step_debug(x, label)
print("Loss:", loss)


===== INPUT =====
x shape: torch.Size([1, 28, 28])

[Conv]
conv_out shape: torch.Size([8, 26, 26])

[ReLU]
relu_out min/max: 0.0 1.463613510131836

[MaxPool]
pool_out shape: torch.Size([8, 13, 13])

[Flatten]
flat shape: torch.Size([1352])

[Linear]
logits shape: torch.Size([10])
logits: tensor([ 0.7248,  0.5729,  2.0574, -0.0431,  0.4604, -0.2800, -0.4742, -0.4830,
         0.9023, -0.3649])

[Softmax]
probs: tensor([0.1066, 0.0916, 0.4042, 0.0495, 0.0819, 0.0390, 0.0321, 0.0319, 0.1273,
        0.0359])
sum of probs (should be 1): 0.9999999403953552

[Loss]
loss: 3.006265163421631

===== BACKWARD =====
dlogits: tensor([ 0.1066,  0.0916,  0.4042, -0.9505,  0.0819,  0.0390,  0.0321,  0.0319,
         0.1273,  0.0359])

[Linear Backward]
dflat shape: torch.Size([1352])

[Flatten Backward]
dpool shape: torch.Size([8, 13, 13])

[MaxPool Backward]
drelu shape: torch.Size([8, 26, 26])

[ReLU Backward]
dconv shape: torch.Size([8, 26, 26])

[Conv Backward]
dx shape: torch.Size([1, 28, 28])
L

In [89]:
import torch

# =========================
# SIMPLE CONV (NO im2col)
# =========================
def conv_forward(x, W, b):
    C, H, W_in = x.shape
    F, _, kH, kW = W.shape

    out_H = H - kH + 1
    out_W = W_in - kW + 1

    out = torch.zeros((F, out_H, out_W))

    for f in range(F):
        for i in range(out_H):
            for j in range(out_W):
                region = x[:, i:i+kH, j:j+kW]
                out[f, i, j] = torch.sum(region * W[f]) + b[f]

    return out


# =========================
# RELU
# =========================
def relu(x):
    return torch.maximum(x, torch.tensor(0.0))


# =========================
# FLATTEN
# =========================
def flatten(x):
    return x.view(-1)


# =========================
# LINEAR
# =========================
def linear(x, W, b):
    return x @ W.T + b


# =========================
# SOFTMAX
# =========================
def softmax(x):
    exp = torch.exp(x - torch.max(x))
    return exp / torch.sum(exp)


# =========================
# LOSS
# =========================
def cross_entropy(probs, label):
    return -torch.log(probs[label])


# =========================
# PARAMETERS
# =========================
conv_W = torch.randn(1, 1, 3, 3) * 0.1   # 1 filter
conv_b = torch.zeros(1)

fc_W = torch.randn(2, 9) * 0.1   # 2 classes, input 3x3=9
fc_b = torch.zeros(2)


# =========================
# FORWARD PASS ONLY
# =========================
def forward_debug(x):

    print("\nINPUT:", x.shape)

    conv = conv_forward(x, conv_W, conv_b)
    print("After Conv:", conv.shape)

    relu_out = relu(conv)
    print("After ReLU:", relu_out.shape)

    flat = flatten(relu_out)
    print("After Flatten:", flat.shape)

    logits = linear(flat, fc_W, fc_b)
    print("After Linear:", logits.shape)

    probs = softmax(logits)
    print("After Softmax:", probs)

    return probs


# =========================
# TEST
# =========================
x = torch.randn(1, 5, 5)   # SMALL IMAGE
label = 1

probs = forward_debug(x)
loss = cross_entropy(probs, label)

print("\nLOSS:", loss.item())


INPUT: torch.Size([1, 5, 5])
After Conv: torch.Size([1, 3, 3])
After ReLU: torch.Size([1, 3, 3])
After Flatten: torch.Size([9])
After Linear: torch.Size([2])
After Softmax: tensor([0.4825, 0.5175])

LOSS: 0.658667802810669
